In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, r2_score, mean_absolute_percentage_error, 
                             mean_squared_error, accuracy_score, precision_score, 
                             recall_score, f1_score)

# Carregamento de Dados
ficheiro = 'transacoes_imobiliarias/SP/Transações ITBI - 2026.xlsx'
xls = pd.ExcelFile(ficheiro)

# Filtrar apenas as abas referentes aos meses e concatenar
abas_meses = [aba for aba in xls.sheet_names if '-' in aba]
df_list = [pd.read_excel(ficheiro, sheet_name=aba) for aba in abas_meses]
df = pd.concat(df_list, ignore_index=True)

# Manter apenas transações de compra e venda
df = df[df['Natureza de Transação'] == '1.Compra e venda'].copy()

# Alvo 1: Valor de Transação (Regressão)
target_reg = 'Valor de Transação (declarado pelo contribuinte)'
df[target_reg] = pd.to_numeric(df[target_reg], errors='coerce')

# Alvo 2: Dependência de Financiamento (Classificação: 1 para financiado, 0 para à vista)
df['Financiado'] = df['Tipo de Financiamento'].notna().astype(int)
target_clf = 'Financiado'

# Converter colunas base para numérico
df['ACC (IPTU)'] = pd.to_numeric(df['ACC (IPTU)'], errors='coerce')
df['Área Construída (m2)'] = pd.to_numeric(df['Área Construída (m2)'], errors='coerce')
df['Área do Terreno (m2)'] = pd.to_numeric(df['Área do Terreno (m2)'], errors='coerce')
df['Valor Venal de Referência'] = pd.to_numeric(df['Valor Venal de Referência'], errors='coerce')

# Criar Flag de Terreno
df['Eh_Terreno'] = (df['Área Construída (m2)'] == 0).astype(int)

# Calcular a Idade do Imóvel (Ano base 2026 - Ano de Construção)
df['Idade_Imovel'] = 2026 - df['ACC (IPTU)']
df.loc[df['Eh_Terreno'] == 1, 'Idade_Imovel'] = 0

# Recuperação de Bairro utilizando o CEP para preencher valores nulos
df['Bairro'] = df['Bairro'].fillna(df.groupby('CEP')['Bairro'].transform('first'))
# Categoria de contingência para os nulos restantes
df['Bairro'] = df['Bairro'].fillna('NAO_INFORMADO')

# Definir as variáveis preditoras
features_numericas = ['Valor Venal de Referência', 'Área Construída (m2)', 'Área do Terreno (m2)', 'Idade_Imovel', 'Eh_Terreno']
features_categoricas = ['Situação do SQL', 'Descrição do uso (IPTU)', 'Bairro']
features = features_numericas + features_categoricas

# Forçar todas as colunas categóricas para string (texto)
for col in features_categoricas:
    df[col] = df[col].astype(str)

# Filtrar nulos no escopo selecionado (como tratámos o Bairro, não perderemos esses registos)
df_model = df[features + [target_reg, target_clf]].dropna()

# Remover extremos para focar no mercado primário
df_model = df_model[(df_model[target_reg] >= 50000) & (df_model[target_reg] <= 5000000)].copy()

# Divisão dos dados em variáveis preditoras (X) e alvo (y)
X = df_model[features]

# Transformação Logarítmica no valor (target_reg)
y_reg_log = np.log1p(df_model[target_reg])
y_clf = df_model[target_clf]

# Divisão em Treino e Teste
X_train, X_test, y_train_reg_log, y_test_reg_log, y_train_clf, y_test_clf = train_test_split(
    X, y_reg_log, y_clf, test_size=0.2, random_state=123
)

# Calcula a média do log do preço para cada bairro no conjunto de TREINO
medias_bairro = y_train_reg_log.groupby(X_train['Bairro']).mean()

# Mapeia as médias para os conjuntos de treino e teste
X_train['Bairro_Encoded'] = X_train['Bairro'].map(medias_bairro)
X_test['Bairro_Encoded'] = X_test['Bairro'].map(medias_bairro)

# Preenche eventuais bairros no teste que não existiam no treino com a média geral
X_test['Bairro_Encoded'] = X_test['Bairro_Encoded'].fillna(y_train_reg_log.mean())

# Remove a coluna de texto original do Bairro
X_train = X_train.drop(columns=['Bairro'])
X_test = X_test.drop(columns=['Bairro'])

# One-Hot Encoding para as variáveis categóricas restantes
features_para_dummies = ['Situação do SQL', 'Descrição do uso (IPTU)']
X_train = pd.get_dummies(X_train, columns=features_para_dummies, drop_first=True)
X_test = pd.get_dummies(X_test, columns=features_para_dummies, drop_first=True)

# Garantir que as colunas do teste são exatamente iguais às do treino após o get_dummies
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Tamanho do conjunto de treino: {len(X_train)} registos")
print(f"Tamanho do conjunto de teste: {len(X_test)} registos\n")

# Treinamento e Avaliação: Modelo de Valor de Transação (Regressão)
modelo_reg = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=123, n_jobs=-1)
modelo_reg.fit(X_train, y_train_reg_log)

previsoes_reg_log = modelo_reg.predict(X_test)
previsoes_reais = np.expm1(previsoes_reg_log)
y_test_real = np.expm1(y_test_reg_log)

mae = mean_absolute_error(y_test_real, previsoes_reais)
rmse = np.sqrt(mean_squared_error(y_test_real, previsoes_reais))
mape = mean_absolute_percentage_error(y_test_real, previsoes_reais)
r2 = r2_score(y_test_real, previsoes_reais)

print("--- RESULTADOS OTIMIZADOS: PREVISÃO DE VALOR (REGRESSÃO) ---")
print(f"R²: {r2:.4f}")
print(f"MAPE: {mape * 100:.2f}%")
print(f"MAE: R$ {mae:,.2f}")
print(f"RMSE: R$ {rmse:,.2f}\n")

# Treinamento e Avaliação: Modelo de Financiamento (Classificação)
modelo_clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=123, n_jobs=-1)
modelo_clf.fit(X_train, y_train_clf)

previsoes_clf = modelo_clf.predict(X_test)

acc = accuracy_score(y_test_clf, previsoes_clf)
prec = precision_score(y_test_clf, previsoes_clf)
rec = recall_score(y_test_clf, previsoes_clf)
f1 = f1_score(y_test_clf, previsoes_clf)

print("--- RESULTADOS: PREVISÃO DE FINANCIAMENTO (CLASSIFICAÇÃO) ---")
print(f"Acurácia: {acc * 100:.2f}%")
print(f"Precisão: {prec * 100:.2f}%")
print(f"Recall:   {rec * 100:.2f}%")
print(f"F1-Score: {f1 * 100:.2f}%")

Tamanho do conjunto de treino: 107122 registos
Tamanho do conjunto de teste: 26781 registos

--- RESULTADOS OTIMIZADOS: PREVISÃO DE VALOR (REGRESSÃO) ---
R²: 0.5880
MAPE: 30.03%
MAE: R$ 153,973.68
RMSE: R$ 365,085.40

--- RESULTADOS: PREVISÃO DE FINANCIAMENTO (CLASSIFICAÇÃO) ---
Acurácia: 77.58%
Precisão: 87.87%
Recall:   70.45%
F1-Score: 78.20%
